### Invocations initiales et chargement des données¶

In [1]:
import os

# Crée un dossier data
dossier_tp = "tpgraphes/myproject/"
fichier_txt_a = dossier_tp + "powergrid.edgelist.txt"
fichier_csv = dossier_tp + "powergrid.csv"

if not os.path.exists(fichier_txt_a):
  os.system("mkdir -p tpgraphes/myproject")


In [2]:
if not os.path.exists(fichier_txt_a):
  os.system("gunzip tpgraphes/myproject/*txt")

In [3]:
if not os.path.exists(fichier_csv):
  os.system("gunzip tpgraphes/myproject/*csv")

In [4]:
!ls -lh tpgraphes/myproject/*

-rw-rw-r-- 1 jovyan jovyan 68K Apr  5 22:37 tpgraphes/myproject/powergrid.csv
-rw-rw-r-- 1 jovyan jovyan 62K Mar 28 14:46 tpgraphes/myproject/powergrid.edgelist.txt


In [5]:
# récupère le lsa.jar
os.system("mkdir -p tpgraphes/jars")
os.system("wget -nv -nc http://cedric.cnam.fr/~ferecatu/RCP216/tp/tptexte/lsa.jar -P tpgraphes/jars")
# récupère spark-xml
os.system("wget -nv -nc http://cedric.cnam.fr/vertigo/Cours/RCP216/docs/spark-xml_2.11-0.9.0.jar -P tpgraphes/jars")

0

In [6]:
import sys
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
os.environ['PYSPARK_SUBMIT_ARGS'] = '--jars /home/jovyan/MonDossier/tpgraphes/jars/spark-xml_2.11-0.9.0.jar --packages graphframes:graphframes:0.8.1-spark3.0-s_2.12 pyspark-shell'

In [7]:
from pyspark.conf import SparkConf
from pyspark.sql.session import SparkSession
from pyspark.context import SparkContext
conf = SparkConf().set("spark.jars", "tpgraphes/jars/lsa.jar, tpgraphes/jars/spark-xml_2.11-0.9.0.jar")
sc = SparkContext.getOrCreate(conf=conf)
spark = SparkSession.builder.getOrCreate()
from pyspark.sql.types import *
import csv

:: loading settings :: url = jar:file:/srv/conda/envs/notebook/lib/python3.8/site-packages/pyspark/jars/ivy-2.5.0.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jovyan/.ivy2/cache
The jars for the packages stored in: /home/jovyan/.ivy2/jars
graphframes#graphframes added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-8b2a656a-feb2-4325-ad83-ec6b5088b5fc;1.0
	confs: [default]
	found graphframes#graphframes;0.8.1-spark3.0-s_2.12 in spark-packages
	found org.slf4j#slf4j-api;1.7.16 in central
downloading https://repos.spark-packages.org/graphframes/graphframes/0.8.1-spark3.0-s_2.12/graphframes-0.8.1-spark3.0-s_2.12.jar ...
	[SUCCESSFUL ] graphframes#graphframes;0.8.1-spark3.0-s_2.12!graphframes.jar (16ms)
downloading https://repo1.maven.org/maven2/org/slf4j/slf4j-api/1.7.16/slf4j-api-1.7.16.jar ...
	[SUCCESSFUL ] org.slf4j#slf4j-api;1.7.16!slf4j-api.jar (33ms)
:: resolution report :: resolve 1153ms :: artifacts dl 53ms
	:: modules in use:
	graphframes#graphframes;0.8.1-spark3.0-s_2.12 from spark-packages in [default]
	org.slf4j#slf4j-api;1.7.16 from central in [default]
	-----------

25/04/08 16:21:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


### Création du graphe

Pour créer un objet graphe, nous avons besoin de deux DataFrame : celui des raaêtes et celui des sommets.
Un DataFrame d’arêtes doit contenir 2 colonnes spéciales, src et dst, pour stocker les identifiants des deux extrémités de chaque arête.
Etant donné que nous disposons dorénavant d'un fichier .csv, créé à partir d'un fichier .txt initial, nous allons en profiter pour créer le DataFrame sur la base de ce fichier .csv

In [8]:
node_fields = [
        StructField("src", StringType(), True),
        StructField("dst", StringType(), True)
        
    ]

rels = spark.read.csv("tpgraphes/myproject/powergrid.csv", header=False, schema=StructType(node_fields))

Vérifions que le DataFrame des relations est créé avec succès

In [9]:
rels.count()

6594

In [10]:
rels.printSchema()

root
 |-- src: string (nullable = true)
 |-- dst: string (nullable = true)



In [11]:
rels.show(5)

+---+----+
|src| dst|
+---+----+
|  0| 386|
|  0| 395|
|  0| 451|
|  1|3553|
|  1|3586|
+---+----+
only showing top 5 rows



Maintenant, il s'agit de créer un deuxième DataFrame : celui des sommets.
Nous ne disposons pas de liste de noeuds, mais nous savons d'après la documentation que "Les sommets sont numérotés à partir de 0, de façon continue". D'après le fichier .csv nous pouvons voir que l'id qui correspond au nombre maximal dans la liste des liens est 4939. Or, le nombre total des noeuds d'après l'analyse que nous avons réalisé dans le logiciel Gephi est de 4773. 
Pour des raisons de praticité, nous avons généré la liste des noeuds comme une suite continue de 4940 nombres entiers de 0 à 4939.
Cependant, nous nous attendons à trouver dans le graphe ainsi créé 167 sommets isolés, qui pourront être triés par la suite : en effet, étant donné que les données comportent uniquement des lignes qui correspondent aux arrêtes entre deux sommets différents du graphe, il n'est pas possible de nous attendre à y trouver de sommets isolés.

In [12]:
nodes = spark.range(0,4940)
nodes.printSchema()
#nodes.show(20)

root
 |-- id: long (nullable = false)



Nous pouvons maintenant importer la bibliothèque GraphFrame et l'utiliser pour créer un objet graphe.

In [13]:
from graphframes import *
g = GraphFrame(nodes, rels)

/srv/conda/envs/notebook/lib/python3.8/site-packages/pyspark/sql/dataframe.py:148: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(


### PREMIER APERCU DU GRAPHE

Vérifions maintenant quelques éléments à propos du graphe créé, comme le nombre de nœuds et d’arêtes

In [14]:
#Calcul du nombre de noeuds
g.vertices.count()

4940

In [15]:
#Calcul du nombre d'arrêtes
g.edges.count()

6594

Nous retrouvons bien 4940 sommets et 6594 arrêtes dans l'objet graphe, ce qui est cohérent avec la documentation, ainsi que les résultats fournis par le logiciel Gephi lors de la première étape de notre travail.

In [16]:
#Aperçu d'un échantillon d'arrêtes
g.edges.show()

+---+----+
|src| dst|
+---+----+
|  0| 386|
|  0| 395|
|  0| 451|
|  1|3553|
|  1|3586|
|  1|3587|
|  1|3637|
|  2|3583|
|  3| 493|
|  4|  88|
|  5|  12|
|  5|  13|
|  6|   8|
|  7|   8|
|  8|   9|
|  9|   1|
|  9| 205|
|  9| 208|
|  9|  61|
|  9|  75|
+---+----+
only showing top 20 rows



### Centralité de Degré

La centralité de degré consiste à ordonner les nœuds en fonction de leur degré, c’est-à-dire du nombre d’autres nœuds qui leur sont liés. 
Dans un contexte de graphe dirigé, on considère éventuellement des degrés entrants et sortants (*out-degree*).
Le graphe donné est de type mixte : d'après la description du fichier initial, "Une ligne contient deux identifiants A et B, ce qui correspond selon les cas à un lien dirigé A->B ou à un lien non dirigé A-B. Les liens non dirigés n'apparaissent qu'une seule fois dans le fichier."
Il n'est donc pas possible de savoir s'il s'agit d'un lien dirigé ou non dirigé pour un lien particulier. 
Nous allons donc calculer les degrés totaux, entrants et extrants, tout en sachant que ces deux deuxièmes paramètres seront probablement erronnés pour certains noeuds.
Nous allons également afficher les ID des noeuds de fort degré :

In [17]:
#Aperçu des degrés entrants et sortants des 40 noeuds ayant le degré total le plus élevé
total_degree = g.degrees
in_degree = g.inDegrees
out_degree = g.outDegrees

(total_degree.join(in_degree, "id", how="left")
.join(out_degree, "id", how="left")
.fillna(0)
.sort("degree", ascending=False)
.show(40))

/srv/conda/envs/notebook/lib/python3.8/site-packages/pyspark/sql/dataframe.py:127: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


+----+------+--------+---------+
|  id|degree|inDegree|outDegree|
+----+------+--------+---------+
|2553|    19|       0|       19|
|4458|    18|      13|        5|
|3468|    14|       6|        8|
|4345|    14|       3|       11|
| 831|    14|       0|       14|
|2542|    13|       2|       11|
|2382|    13|       7|        6|
|3895|    13|       9|        4|
|2575|    13|       2|       11|
|2585|    13|       0|       13|
|  14|    13|       9|        4|
|2439|    12|       2|       10|
|2434|    12|       0|       12|
|2662|    12|       1|       11|
|2617|    12|       7|        5|
|1224|    12|       3|        9|
|4384|    11|       4|        7|
|4381|    11|       8|        3|
| 402|    11|       6|        5|
|1334|    11|       4|        7|
|4395|    11|      10|        1|
|1005|    11|       3|        8|
| 490|    11|       0|       11|
| 313|    11|      11|        0|
|4332|    11|       0|       11|
|  35|    11|       7|        4|
|2282|    11|       5|        6|
|4373|    

In [18]:
#Aperçu des degrés entrants et sortants des 20 noeuds ayant le degré total le plus élevé
total_degree = g.degrees
in_degree = g.inDegrees
out_degree = g.outDegrees

(total_degree.join(in_degree, "id", how="left")
.join(out_degree, "id", how="left")
.fillna(0)
.sort("degree", ascending=False)
.show(20))

+----+------+--------+---------+
|  id|degree|inDegree|outDegree|
+----+------+--------+---------+
|2553|    19|       0|       19|
|4458|    18|      13|        5|
|3468|    14|       6|        8|
|4345|    14|       3|       11|
| 831|    14|       0|       14|
|2575|    13|       2|       11|
|2382|    13|       7|        6|
|2542|    13|       2|       11|
|3895|    13|       9|        4|
|2585|    13|       0|       13|
|  14|    13|       9|        4|
|2439|    12|       2|       10|
|2434|    12|       0|       12|
|2617|    12|       7|        5|
|2662|    12|       1|       11|
|1224|    12|       3|        9|
|2282|    11|       5|        6|
|4395|    11|      10|        1|
|4381|    11|       8|        3|
| 490|    11|       0|       11|
+----+------+--------+---------+
only showing top 20 rows



Analysons maintenant la distribution des degrés des noeuds du graphe.
Affichons d'abord les statistiques générales.

In [19]:
degrees = g.degrees
degrees.describe('degree').show()

+-------+------------------+
|summary|            degree|
+-------+------------------+
|  count|              4773|
|   mean|2.7630421118793214|
| stddev|1.8807156877148299|
|    min|                 1|
|    max|                19|
+-------+------------------+



Nous pouvons ainsi constater que le degré maximal des noeuds du graphe est égal à 19, le degré maximal est égal à 19.
Le degré moyen est égal à 2.76, avec un écart-type de 1.88. Ces statistiques ont été calculés pour les 4773 noeuds reliés à au moins un autre noeud du graphe.
Nous constatons que la moyenne des degrés est très faible par rapport au degré maximal du graphe.
Affichons maintenant le nombre de noeuds pour chacune des valeurs de degré total observé dans ce graphe.

In [20]:
d = g.degrees.select("degree").rdd.flatMap(lambda x:x)
freqDegres = d.countByValue()
# on obtient le nombre de sommets de degrés n
distribDegres = sorted(freqDegres.items(), key=lambda item: item[0], reverse=True)

print("degré","nb_sommets")
for k,v in enumerate(distribDegres[:17]):
    print(v[0],v[1])

degré nb_sommets
19 1
18 1
14 3
13 6
12 5
11 17
10 26
9 34
8 42
7 95
6 160
5 256
4 451
3 1007
2 1483
1 1186


Nous observons ainsi deux noeuds "grands leaders", à savoir les noeuds 2553 et 4458, dont le degré total est égal à 19 et 18 respectivement.
Ces noeuds importants correpondent probablement aux points nevralgiques les plus importants du réseau électrique (comme une centrale de production par exemple).
Ces deux grands leaders sont suivis par d'autres noeuds à forte importance, à savoir 
3 noeuds de degré 14 (3468, 4345 et 831);
6 noeuds de degré 13;
5 noeuds de degré 12. 
Pour les valeurs de degré de 11 à 7, on trouve des valeurs de nombre de noeuds supérieures à 10. 
Pour les valeurs de degré de 6 à 4, on trouve des nombres de noeuds supérieures à 100.
Enfin, la grande majorité des noeuds du réseau (à savoir plus de 3500 noeuds) sont caractérisé par un faible degré (entre 3 et 1) : ces noeuds correpondent probablement aux points de consommation de l'électricité.

### ANALYSE DE LA CONNEXITE ET IDENTIFICATION DES COMPOSANTES CONNEXES

In [21]:
!mkdir checkpoints1
spark.sparkContext.setCheckpointDir('checkpoints1')

mkdir: cannot create directory ‘checkpoints1’: File exists


Nous allons maintenant  regarder la connexité du graphe. Un graphe est dit connexe lorsqu'il est possible d’atteindre un sommet depuis n’importe quel autre en suivant un chemin (une séquence de sommets). Un graphe qui n’est pas connexe est divisé en plusieurs composantes connexes, c'est-à-dire des sous-parties du graphes qui peuvent être examinées individuellement. 

In [22]:
# calcul des composantes connexes du graphe
gCC = g.connectedComponents()


Le dataframe gCC contient les identifiants des sommets du graphe, un hash et la composante connexe à laquelle ils appartiennent.
La méthode suivante, destinées à afficher les composantes connexes triées par taille, peut échouer, compte tenu de sa complexité et de la taille des structures manipulées.
Dans le cas présent, elle ne permet que d'afficher la première composante connexe géante, mais pas les composantes connexes suivantes.

In [25]:
from pyspark.sql import functions as F

(gCC.sort("component")
       .groupby("component")
       .agg(F.collect_list("id").alias("libraries"))
       .show(8,truncate=False))

+---------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

Nous allons maintenant compter le nombre de composantes connexes puis de les afficher triées par taille :


In [24]:
gCCrdd = gCC.select('component').rdd
from pyspark.sql import Row
temp = gCCrdd.map(lambda row: row['component'])
cc = temp.countByValue()

cc_sorted = sorted(cc.items(), key=lambda item: item[1], reverse=True)
for k,v in enumerate(cc_sorted[:60]):
    print("Rang", k+1, ": composante connexe", v[0], "nb noeuds:", v[1])

Rang 1 : composante connexe 0 nb noeuds: 4723
Rang 2 : composante connexe 2747 nb noeuds: 5
Rang 3 : composante connexe 4210 nb noeuds: 5
Rang 4 : composante connexe 4530 nb noeuds: 4
Rang 5 : composante connexe 1760 nb noeuds: 3
Rang 6 : composante connexe 4550 nb noeuds: 3
Rang 7 : composante connexe 530 nb noeuds: 2
Rang 8 : composante connexe 640 nb noeuds: 2
Rang 9 : composante connexe 1780 nb noeuds: 2
Rang 10 : composante connexe 2180 nb noeuds: 2
Rang 11 : composante connexe 2590 nb noeuds: 2
Rang 12 : composante connexe 2600 nb noeuds: 2
Rang 13 : composante connexe 2730 nb noeuds: 2
Rang 14 : composante connexe 2860 nb noeuds: 2
Rang 15 : composante connexe 2870 nb noeuds: 2
Rang 16 : composante connexe 2900 nb noeuds: 2
Rang 17 : composante connexe 2960 nb noeuds: 2
Rang 18 : composante connexe 2970 nb noeuds: 2
Rang 19 : composante connexe 3590 nb noeuds: 2
Rang 20 : composante connexe 4620 nb noeuds: 2
Rang 21 : composante connexe 4780 nb noeuds: 2
Rang 22 : composante con

Nous constatons ainsi que le graphe compte 21 composante connexe d'au moins 2 noeuds (comme indiqué plus haut, nous ignorons dans notre analyse les noeuds isolés qui correspondent probablement aux ID non présents dans le fichier des arrêtes).
La première composante connexte qui se détache largement par sa taille par rapport aux suivantes, contient 4722 noeuds, c'est-à-dire la grande majorité des noeuds du réseau.
Cette composante connexe principale est suivie d'une multitude de petites composantes connexes de petite taille :
2 composantes connexes de 5 noeuds 
1 de 4 noeuds
2 de 3 noeuds
15 de 2 noeuds.
Enfin, on trouve également de noeuds isolés : compte tenu de la démarche que nous avons adoptée lors de la création de l'objet DataFrame, ce sont sûrement les ID qui ont été générés lors de la création du DataFrame des sommets mais qui n'étaient pas présents dans le fichier des arrêtes.
Nous pouvons donc ignorer la présence de ces sommets isolés dans notre analyse.

Regardons maintenant de plus près les ID des sommets qui composent les composantes connexes de petite taille

In [25]:
for k,v in enumerate(cc_sorted[1:21]):
    print("Rang", k+1, ": composante connexe", v[0], "nb noeuds:", v[1])
    print(gCC.filter(gCC.component==v[0]).select(gCC.id).show(5,False))

Rang 1 : composante connexe 2747 nb noeuds: 5
+----+
|id  |
+----+
|2747|
|3031|
|3082|
|3090|
|3101|
+----+

None
Rang 2 : composante connexe 4210 nb noeuds: 5
+----+
|id  |
+----+
|4210|
|4313|
|4314|
|4315|
|4316|
+----+

None
Rang 3 : composante connexe 4530 nb noeuds: 4
+----+
|id  |
+----+
|4530|
|4590|
|4657|
|4687|
+----+

None
Rang 4 : composante connexe 1760 nb noeuds: 3
+----+
|id  |
+----+
|1760|
|1898|
|2088|
+----+

None
Rang 5 : composante connexe 4550 nb noeuds: 3
+----+
|id  |
+----+
|4550|
|4600|
|4624|
+----+

None
Rang 6 : composante connexe 530 nb noeuds: 2
+---+
|id |
+---+
|530|
|531|
+---+

None
Rang 7 : composante connexe 640 nb noeuds: 2
+---+
|id |
+---+
|640|
|641|
+---+

None
Rang 8 : composante connexe 1780 nb noeuds: 2
+----+
|id  |
+----+
|1780|
|1947|
+----+

None
Rang 9 : composante connexe 2180 nb noeuds: 2
+----+
|id  |
+----+
|2180|
|2181|
+----+

None
Rang 10 : composante connexe 2590 nb noeuds: 2
+----+
|id  |
+----+
|2590|
|2912|
+----+

None
Ran

### PageRank

Approche avec nombre d’itérations fixé

In [26]:
results = g.pageRank(resetProbability=0.15, maxIter=20)
results.vertices.sort("pagerank", ascending=False).show(35)

+----+------------------+
|  id|          pagerank|
+----+------------------+
|4458| 9.942269477227606|
| 207|  9.81285069213353|
|  14| 9.101335378819842|
| 193| 8.498424411244196|
|4461| 8.403592457057025|
| 171|6.6867800211663715|
| 205| 6.625602537643494|
| 362| 6.616774920536669|
| 337| 6.496684668413614|
| 408|6.4802824519553495|
|4446| 6.257328038653206|
| 352| 6.249628055757778|
| 201| 6.053970947325331|
|1057|6.0408677933689505|
| 313| 5.920161235981189|
|   2| 5.832927906432979|
|3625| 5.731066841178158|
| 184|5.5296990064798495|
| 237| 5.524084565754929|
|3583| 5.516776130153721|
|  98| 5.510940530886086|
|4199| 5.496779999383956|
| 466| 5.449270106355892|
| 726| 5.418534654544597|
|  47|   5.4084130246243|
|3781| 5.397106870833036|
|   9|5.3728153798342495|
|  13| 5.256726906046148|
| 494| 5.235935200141085|
|4448| 5.154944224664988|
|4396| 5.117498538978598|
|4402| 5.062055559139965|
| 204| 4.929300301860529|
| 316| 4.732030141582738|
|4443| 4.710210727925541|
+----+------

Approche « jusqu’à convergence »

In [36]:
results2 = g.pageRank(resetProbability=0.15, tol=0.01)
results2.vertices.sort("pagerank", ascending=False).show(35)

+----+------------------+
|  id|          pagerank|
+----+------------------+
|4458| 9.908077586750819|
| 207| 9.774762049324144|
|  14| 8.990653481862342|
| 193| 8.409252633603955|
|4461| 8.352729428715351|
| 362| 6.608084004500827|
| 171| 6.584067768550585|
| 205|6.5757570026306595|
| 408| 6.482783450795807|
| 337| 6.443064607645123|
| 352| 6.229443630550263|
|4446|  6.20951341762516|
| 201| 6.039237284069965|
|1057|  5.98121058970385|
| 313| 5.876154258547248|
|   2|5.7890420025383715|
|3625| 5.680107646467643|
| 237| 5.511896091019855|
| 184| 5.503289220057882|
|4199|  5.49480151067667|
|3583| 5.481390980570609|
|  98| 5.452533656954471|
| 466| 5.432898779452955|
| 726|5.3882047028289435|
|3781| 5.374042489620366|
|  47|  5.33972229806208|
|   9| 5.315515271961476|
| 494|5.2163037778976475|
|  13|5.1822507015363515|
|4448|  5.12273155009707|
|4396| 5.075639860733814|
|4402| 5.035355555983041|
| 204| 4.938383091632372|
|4443| 4.696943582121204|
| 316| 4.691848295319803|
+----+------

### Centralité de Proximité

In [17]:
from graphframes.lib import AggregateMessages as AM
from pyspark.sql import functions as F
from pyspark.sql.types import *
from operator import itemgetter

In [18]:
def collect_paths(paths):
    return F.collect_set(paths)

collect_paths_udf = F.udf(collect_paths, ArrayType(StringType()))

paths_type = ArrayType(
    StructType([StructField("id", StringType()), StructField("distance", IntegerType())]))

def flatten(ids):
    flat_list = [item for sublist in ids for item in sublist]
    return list(dict(sorted(flat_list, key=itemgetter(0))).items())

flatten_udf = F.udf(flatten, paths_type)

def new_paths(paths, id):
    paths = [{"id": col1, "distance": col2 + 1} for col1, col2 in paths if col1 != id]
    paths.append({"id": id, "distance": 1})
    return paths

new_paths_udf = F.udf(new_paths, paths_type)

def merge_paths(ids, new_ids, id):
    joined_ids = ids + (new_ids if new_ids else [])
    merged_ids = [(col1, col2) for col1, col2 in joined_ids if col1 != id]
    best_ids = dict(sorted(merged_ids, key=itemgetter(1), reverse=True))
    return [{"id": col1, "distance": col2} for col1, col2 in best_ids.items()]

merge_paths_udf = F.udf(merge_paths, paths_type)

def calculate_closeness(ids):
    nodes = len(ids)
    total_distance = sum([col2 for col1, col2 in ids])
    return 0 if total_distance == 0 else nodes * 1.0 / total_distance

closeness_udf = F.udf(calculate_closeness, DoubleType())

La méthode suivante a échoué malgré de plusieurs tentatives : 

Exception in thread "netty-rpc-env-timeout" java.lang.OutOfMemoryError: GC overhead limit exceeded

25/04/07 21:19:05 ERROR Utils: Uncaught exception in thread executor-heartbeater
java.lang.OutOfMemoryError: GC overhead limit exceeded

In [19]:
vertices = g.vertices.withColumn("ids", F.array())
cached_vertices = AM.getCachedDataFrame(vertices)
g2 = GraphFrame(cached_vertices, g.edges)

for i in range(0, g2.vertices.count()):
    msg_dst = new_paths_udf(AM.src["ids"], AM.src["id"])
    msg_src = new_paths_udf(AM.dst["ids"], AM.dst["id"])
    agg = g2.aggregateMessages(F.collect_set(AM.msg).alias("agg"),
                               sendToSrc=msg_src, sendToDst=msg_dst)
    res = agg.withColumn("newIds", flatten_udf("agg")).drop("agg")
    new_vertices = (g2.vertices.join(res, on="id", how="left_outer")
                    .withColumn("mergedIds", merge_paths_udf("ids", "newIds", "id"))
                    .drop("ids", "newIds")
                    .withColumnRenamed("mergedIds", "ids"))
    cached_new_vertices = AM.getCachedDataFrame(new_vertices)
    g2 = GraphFrame(cached_new_vertices, g2.edges)

(g2.vertices
 .withColumn("closeness", closeness_udf("ids"))
 .sort("closeness", ascending=False).select('id','closeness')
 .show(10))

/srv/conda/envs/notebook/lib/python3.8/site-packages/pyspark/sql/dataframe.py:127: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")
Exception in thread "refresh progress"                            (0 + 10) / 11]Exception in thread "stdout writer for /srv/conda/envs/notebook/bin/python" java.lang.BootstrapMethodError: call site initialization exception
	at java.lang.invoke.CallSite.makeSite(CallSite.java:341)
	at java.lang.invoke.MethodHandleNatives.linkCallSiteImpl(MethodHandleNatives.java:307)
	at java.lang.invoke.MethodHandleNatives.linkCallSite(MethodHandleNatives.java:297)
	at org.apache.spark.util.Utils$.logUncaughtExceptions(Utils.scala:2071)
	at org.apache.spark.api.python.BasePythonRunner$WriterThread.run(PythonRunner.scala:272)
Caused by: java.lang.OutOfMemoryError: GC overhead limit exceeded
java.lang.OutOfMemoryError: GC overhead limit exceeded
Exception in thread "stdout wr

25/04/07 21:18:19 ERROR Utils: Uncaught exception in thread stdout writer for /srv/conda/envs/notebook/bin/python
java.lang.OutOfMemoryError: GC overhead limit exceeded
25/04/07 21:18:19 ERROR Utils: Uncaught exception in thread driver-heartbeater
java.lang.OutOfMemoryError: GC overhead limit exceeded
25/04/07 21:18:21 ERROR Utils: Uncaught exception in thread stdout writer for /srv/conda/envs/notebook/bin/python
java.lang.OutOfMemoryError: GC overhead limit exceeded
25/04/07 21:18:19 ERROR Utils: Uncaught exception in thread stdout writer for /srv/conda/envs/notebook/bin/python
java.lang.OutOfMemoryError: GC overhead limit exceeded
	at java.lang.Integer.valueOf(Integer.java:832)
	at net.razorvine.pickle.Pickler.lookupMemo(Pickler.java:228)
	at net.razorvine.pickle.Pickler.save(Pickler.java:185)
	at org.apache.spark.sql.execution.python.EvaluatePython$RowPickler.pickle(EvaluatePython.scala:265)
	at net.razorvine.pickle.Pickler.dispatch(Pickler.java:297)
	at net.razorvine.pickle.Pickler

Exception in thread "stdout writer for /srv/conda/envs/notebook/bin/python" java.lang.OutOfMemoryError: GC overhead limit exceeded
Exception in thread "stdout writer for /srv/conda/envs/notebook/bin/python" java.lang.OutOfMemoryError: GC overhead limit exceeded
	at java.lang.Integer.valueOf(Integer.java:832)
	at net.razorvine.pickle.Pickler.lookupMemo(Pickler.java:228)
	at net.razorvine.pickle.Pickler.save(Pickler.java:185)
	at org.apache.spark.sql.execution.python.EvaluatePython$RowPickler.pickle(EvaluatePython.scala:265)
	at net.razorvine.pickle.Pickler.dispatch(Pickler.java:297)
	at net.razorvine.pickle.Pickler.save(Pickler.java:185)
	at net.razorvine.pickle.Pickler.put_collection(Pickler.java:407)
	at net.razorvine.pickle.Pickler.dispatch(Pickler.java:363)
	at net.razorvine.pickle.Pickler.save(Pickler.java:185)
	at net.razorvine.pickle.Pickler.put_collection(Pickler.java:407)
	at net.razorvine.pickle.Pickler.dispatch(Pickler.java:363)
	at net.razorvine.pickle.Pickler.save(Pickler.j

25/04/07 21:18:31 ERROR Utils: Uncaught exception in thread stdout writer for /srv/conda/envs/notebook/bin/python
java.lang.OutOfMemoryError: GC overhead limit exceeded
25/04/07 21:18:31 ERROR Utils: Uncaught exception in thread stdout writer for /srv/conda/envs/notebook/bin/python
java.lang.OutOfMemoryError: GC overhead limit exceeded


Exception in thread "stdout writer for /srv/conda/envs/notebook/bin/python" java.lang.OutOfMemoryError: GC overhead limit exceeded
Exception in thread "stdout writer for /srv/conda/envs/notebook/bin/python" java.lang.OutOfMemoryError: GC overhead limit exceeded


...

25/04/07 21:40:46 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 1307223 ms exceeds timeout 120000 ms
25/04/07 21:40:47 ERROR Inbox: An error happened while processing message in the inbox for HeartbeatReceiver
java.lang.OutOfMemoryError: GC overhead limit exceeded


Exception in thread "dispatcher-event-loop-19" java.lang.OutOfMemoryError: GC overhead limit exceeded


25/04/07 21:41:54 ERROR Inbox: An error happened while processing message in the inbox for HeartbeatReceiver
java.lang.OutOfMemoryError: GC overhead limit exceeded


Exception in thread "dispatcher-event-loop-20" java.lang.OutOfMemoryError: GC overhead limit exceeded
ERROR:root:KeyboardInterrupt while sending command.
Traceback (most recent call last):
  File "/srv/conda/envs/notebook/lib/python3.8/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/srv/conda/envs/notebook/lib/python3.8/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/srv/conda/envs/notebook/lib/python3.8/socket.py", line 669, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

In [20]:
help(GraphFrame)

Help on class GraphFrame in module graphframes.graphframe:

class GraphFrame(builtins.object)
 |  GraphFrame(v, e)
 |  
 |  Represents a graph with vertices and edges stored as DataFrames.
 |  
 |  :param v:  :class:`DataFrame` holding vertex information.
 |             Must contain a column named "id" that stores unique
 |             vertex IDs.
 |  :param e:  :class:`DataFrame` holding edge information.
 |             Must contain two columns "src" and "dst" storing source
 |             vertex IDs and destination vertex IDs of edges, respectively.
 |  
 |  >>> localVertices = [(1,"A"), (2,"B"), (3, "C")]
 |  >>> localEdges = [(1,2,"love"), (2,1,"hate"), (2,3,"follow")]
 |  >>> v = sqlContext.createDataFrame(localVertices, ["id", "name"])
 |  >>> e = sqlContext.createDataFrame(localEdges, ["src", "dst", "action"])
 |  >>> g = GraphFrame(v, e)
 |  
 |  Methods defined here:
 |  
 |  __init__(self, v, e)
 |      Initialize self.  See help(type(self)) for accurate signature.
 |  
 |  _